# Final Test-Set Evaluation And XAI For Saved Models

This Colab notebook evaluates the relevant saved MM-IMDb models on the held-out test split and runs XAI analysis for each selected model.

It covers:
- best overall model selection from evaluated test metrics,
- best classic model,
- best saved neural model,
- ablation models from model selection (`selection_eligible=false`, e.g. text-only and image-only),
- full test-set metrics and predictions,
- global plots and local qualitative XAI artifacts,
- a saved manifest showing exactly which checkpoint/model was used.

Outputs are saved under `outputs/colab_test_xai_all_models/<RUN_NAME>/` and the notebook prints those paths after each stage.

In [ ]:
# Colab setup: clone/update the project and install only missing extras
from google.colab import drive
from pathlib import Path
import sys

REPO_URL = 'https://github.com/SamuelSulan/Explaining-Predictions-of-Machine-Learning-Models.git'
PROJECT_DIR = Path('/content/drive/MyDrive/Explaining-Predictions-of-Machine-Learning-Models')

drive.mount('/content/drive')

if not PROJECT_DIR.exists():
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd $PROJECT_DIR
    !git pull --ff-only

%cd $PROJECT_DIR

# Keep Colab's preinstalled NumPy/Pandas/Torch/Torchvision stack. Installing the
# full requirements file can replace binary packages and create resolver conflicts.
!pip -q install --upgrade --no-deps iterative-stratification captum pypdf
!pip -q install -e . --no-deps

sys.path.insert(0, str(PROJECT_DIR / 'src'))

import numpy as np
import pandas as pd
import torch
import torchvision
import seaborn as sns

print('Project:', PROJECT_DIR)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('Seaborn:', sns.__version__)
print('Dependencies are ready. Continue with the next cells.')

In [ ]:
# User-adjustable controls
from datetime import datetime

RUN_NAME = datetime.now().strftime('run_%Y%m%d_%H%M%S')
OUTPUT_ROOT = Path('outputs/colab_test_xai_all_models') / RUN_NAME

# Set this to False when you only want to reuse saved models and run local XAI examples.
# When False, the notebook skips the full test-set metric/prediction pass and metric plots.
RUN_TEST_EVALUATION = True

# XAI is intentionally sample-limited because Integrated Gradients / occlusion can be slow.
RUN_XAI = True
XAI_LIMIT = 10
XAI_SAMPLING_STRATEGY = 'balanced_support_bins'  # one of: 'balanced_support_bins', 'balanced_per_label', 'random', 'first'
XAI_RANDOM_SEED = 42
XAI_MODEL_KEYS = None  # e.g. ['best_neural'] to run XAI for only selected models; None = all discovered models

XAI_TARGET_POLICY = 'predicted'   # one of: 'top', 'top_k', 'predicted', 'all'
XAI_TARGET_TOP_K = 3
XAI_MAX_TARGETS_PER_SAMPLE = 3
XAI_TOP_FEATURES = 12
XAI_N_STEPS = 8
XAI_IMAGE_OCCLUSION_GRID = 4
XAI_OCCLUSION_BATCH_SIZE = 32

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('All outputs will be saved under:', OUTPUT_ROOT.resolve())
print('RUN_TEST_EVALUATION:', RUN_TEST_EVALUATION)
print('RUN_XAI:', RUN_XAI)
print('XAI sampling:', XAI_SAMPLING_STRATEGY, 'limit=', XAI_LIMIT, 'seed=', XAI_RANDOM_SEED)

In [ ]:
import json
import shutil
from pathlib import Path
from pprint import pprint

import h5py
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse

from mmimdb.constants import GENRE_LABELS
from mmimdb.data import DatasetPaths, load_labels
from mmimdb.evaluation import multilabel_metrics, threshold_predictions, threshold_to_serializable
from mmimdb.final_analysis import _classic_feature_matrix, summarize_xai_outputs
from mmimdb.models.neural import MMIMDBTorchDataset, make_loader, predict
from mmimdb.splits import load_split_indices
from mmimdb.utils import ensure_dir, load_config, resolve_path, save_json
from mmimdb.xai import (
    explain_classic_samples,
    explain_neural_samples,
    load_neural_model,
    patch_classic_classifier_compat,
)

config = load_config('configs/default.yaml')
paths = DatasetPaths.from_config(config)
train_idx, val_idx, test_idx = load_split_indices(resolve_path(config['splits']['output_dir']))
y_all = load_labels(paths.hdf5)

print('Split sizes:', {'train': len(train_idx), 'val': len(val_idx), 'test': len(test_idx)})
print('Dataset paths:', paths)

In [ ]:
def project_path_from_any(raw_path):
    """Resolve Colab/Drive paths and paths copied from local/old runs."""
    if raw_path is None:
        return None
    path = Path(str(raw_path))
    if path.exists():
        return path
    text = str(raw_path).replace('\\', '/')
    marker = '/outputs/'
    if marker in text:
        candidate = Path('outputs') / text.split(marker, 1)[1]
        if candidate.exists():
            return candidate
    if text.startswith('outputs/'):
        candidate = Path(text)
        if candidate.exists():
            return candidate
    return path


def load_json_if_exists(path):
    path = project_path_from_any(path)
    if path and path.exists():
        with path.open('r', encoding='utf-8') as f:
            return json.load(f)
    return None


def add_model(models, key, model_type, path, role, source, notes='', priority=50):
    path = project_path_from_any(path)
    if path is None or not path.exists():
        return
    if key in models:
        # Prefer canonical/final entries over fold fallbacks.
        if priority >= models[key].get('priority', 0):
            return
    models[key] = {
        'key': key,
        'model_type': model_type,
        'path': str(path),
        'role': role,
        'source': source,
        'notes': notes,
        'priority': priority,
    }


def best_fold_for_candidate(candidate):
    folds = [fold for fold in candidate.get('folds', []) if 'score' in fold and fold.get('model_path')]
    if not folds:
        return None
    return max(folds, key=lambda fold: float(fold['score']))


def discover_relevant_models():
    models = {}
    add_model(
        models,
        key='best_classic',
        model_type='classic',
        path='outputs/models/best/classic_multimodal_best.joblib',
        role='best_classic',
        source='canonical best classic registry',
        priority=1,
    )
    add_model(
        models,
        key='best_neural',
        model_type='neural',
        path='outputs/models/best/neural_multimodal_best.pt',
        role='best_neural',
        source='canonical best neural registry',
        priority=2,
    )

    summary = load_json_if_exists('outputs/model_selection/training_summary.json')
    if summary:
        overall = summary.get('overall_best', {})
        if overall.get('model_path'):
            kind = overall.get('kind', 'neural')
            add_model(
                models,
                key='training_summary_overall_best',
                model_type='classic' if kind == 'classic' else 'neural',
                path=overall.get('model_path'),
                role='training_summary_overall_best',
                source='outputs/model_selection/training_summary.json',
                notes=f"Selected by training summary using {overall.get('metric')}={overall.get('score')}",
                priority=3,
            )
        for kind, result in summary.get('kinds', {}).items():
            final = result.get('final', {})
            if final.get('model_path'):
                add_model(
                    models,
                    key=f'final_{kind}',
                    model_type='classic' if kind == 'classic' else 'neural',
                    path=final.get('model_path'),
                    role=f'final_{kind}',
                    source='model_selection final result',
                    priority=8,
                )
            for candidate in result.get('candidates', []):
                params = candidate.get('params', {})
                modality = str(params.get('modality', '')).lower()
                is_ablation = candidate.get('selection_eligible') is False or modality in {'text_only', 'image_only'}
                if not is_ablation:
                    continue
                fold = best_fold_for_candidate(candidate)
                if not fold:
                    continue
                candidate_key = candidate.get('name', 'candidate')
                add_model(
                    models,
                    key=f'ablation_{candidate_key}',
                    model_type='classic' if kind == 'classic' else 'neural',
                    path=fold.get('model_path'),
                    role='ablation',
                    source='best validation fold from model_selection candidate',
                    notes=(
                        f"Candidate {candidate_key}; modality={modality or 'n/a'}; "
                        f"best fold={fold.get('fold')} validation score={fold.get('score')}"
                    ),
                    priority=20,
                )
    deduped = {}
    for entry in sorted(models.values(), key=lambda item: item.get('priority', 50)):
        path_key = str(project_path_from_any(entry['path']).resolve())
        if path_key not in deduped:
            deduped[path_key] = entry
    return list(deduped.values())

model_manifest = discover_relevant_models()
if not model_manifest:
    raise FileNotFoundError('No relevant saved models were found. Check outputs/models/best and outputs/model_selection.')

manifest_path = OUTPUT_ROOT / 'model_manifest.json'
save_json(model_manifest, manifest_path)
print('Discovered relevant models:')
pprint(model_manifest)
print('\nSaved manifest:', manifest_path.resolve())

In [ ]:
import torch


def evaluate_classic_entry(entry):
    artifact = joblib.load(entry['path'])
    patch_classic_classifier_compat(artifact['classifier'])
    x_test = _classic_feature_matrix(artifact, paths.hdf5, paths.metadata, test_idx)
    y_true = y_all[test_idx]
    y_prob = np.asarray(artifact['classifier'].predict_proba(x_test), dtype=np.float32)
    threshold = artifact.get('threshold', 0.5)
    y_pred = threshold_predictions(y_prob, threshold)
    metrics = multilabel_metrics(y_true, y_prob, threshold=threshold)
    return y_true, y_prob, y_pred, threshold, metrics, {'config': artifact.get('config', {})}


def evaluate_neural_entry(entry):
    model, cfg, metadata, checkpoint, device = load_neural_model(entry['path'], paths.metadata)
    dataset = MMIMDBTorchDataset(
        paths.hdf5,
        test_idx,
        vocab_size=int(metadata['vocab_size']),
        max_length=cfg.max_length,
        input_size=cfg.input_size,
    )
    loader = make_loader(dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    y_true, y_prob = predict(model, loader, device)
    threshold = checkpoint.get('threshold', 0.5)
    y_pred = threshold_predictions(y_prob, threshold)
    metrics = multilabel_metrics(y_true, y_prob, threshold=threshold)
    extra = {'config': dict(checkpoint.get('config', {})), 'device': device, 'best_epoch': checkpoint.get('best_epoch')}
    return y_true, y_prob, y_pred, threshold, metrics, extra


def compact_metrics(metrics):
    keys = ['sample_f1', 'micro_f1', 'macro_f1', 'weighted_f1', 'micro_precision', 'micro_recall', 'hamming_loss']
    return {key: float(metrics[key]) for key in keys if key in metrics}

metrics_dir = OUTPUT_ROOT / 'metrics'
pred_dir = OUTPUT_ROOT / 'predictions'
metrics_dir.mkdir(parents=True, exist_ok=True)
pred_dir.mkdir(parents=True, exist_ok=True)

results = []
if RUN_TEST_EVALUATION:
    for entry in model_manifest:
        print(f"\nEvaluating {entry['key']} ({entry['model_type']})")
        if entry['model_type'] == 'classic':
            y_true, y_prob, y_pred, threshold, metrics, extra = evaluate_classic_entry(entry)
        else:
            y_true, y_prob, y_pred, threshold, metrics, extra = evaluate_neural_entry(entry)

        npz_path = pred_dir / f"{entry['key']}_test_predictions.npz"
        np.savez_compressed(npz_path, indices=test_idx, y_true=y_true, y_prob=y_prob, y_pred=y_pred)

        rows = []
        for row_i, dataset_idx in enumerate(test_idx):
            top = np.argsort(y_prob[row_i])[::-1][:5]
            rows.append({
                'dataset_index': int(dataset_idx),
                'true_genres': '|'.join(GENRE_LABELS[i] for i in np.flatnonzero(y_true[row_i])),
                'predicted_genres': '|'.join(GENRE_LABELS[i] for i in np.flatnonzero(y_pred[row_i])),
                'top5_genres': '|'.join(GENRE_LABELS[i] for i in top),
                'top5_probabilities': '|'.join(f'{float(y_prob[row_i, i]):.6f}' for i in top),
            })
        csv_path = pred_dir / f"{entry['key']}_test_predictions.csv"
        pd.DataFrame(rows).to_csv(csv_path, index=False)

        payload = {
            **entry,
            'threshold': threshold_to_serializable(threshold),
            'n_test': int(len(test_idx)),
            'metrics': metrics,
            'prediction_npz': str(npz_path),
            'prediction_csv': str(csv_path),
            'extra': extra,
        }
        metrics_path = metrics_dir / f"{entry['key']}_test_metrics.json"
        save_json(payload, metrics_path)
        results.append(payload)
        print('Metrics:', compact_metrics(metrics))
        print('Saved:', metrics_path.resolve())
        print('Predictions:', csv_path.resolve())

    results_path = OUTPUT_ROOT / 'all_model_test_metrics.json'
    save_json(results, results_path)
    print('\nSaved combined metrics:', results_path.resolve())
else:
    print('RUN_TEST_EVALUATION=False, skipped full test-set metrics and prediction export.')
    print('XAI can still run from model_manifest and the configured test split.')

In [ ]:
# Rank models and identify best overall / best classic / ablations.
metric_name = config.get('model_registry', {}).get('metric', 'macro_f1')

if results:
    ranked = sorted(results, key=lambda r: float(r['metrics'][metric_name]), reverse=True)
    best_overall = ranked[0]
    best_classic = max([r for r in results if r['model_type'] == 'classic'], key=lambda r: float(r['metrics'][metric_name]), default=None)
    ablation_results = [r for r in results if r['role'] == 'ablation']

    selection_summary = {
        'selection_metric': metric_name,
        'best_overall': {'key': best_overall['key'], 'model_type': best_overall['model_type'], 'score': best_overall['metrics'][metric_name], 'path': best_overall['path']},
        'best_classic': None if best_classic is None else {'key': best_classic['key'], 'score': best_classic['metrics'][metric_name], 'path': best_classic['path']},
        'ablations': [{'key': r['key'], 'score': r['metrics'][metric_name], 'path': r['path'], 'notes': r.get('notes', '')} for r in ablation_results],
    }
    save_json(selection_summary, OUTPUT_ROOT / 'selection_summary.json')

    rows = []
    for r in ranked:
        row = {'key': r['key'], 'model_type': r['model_type'], 'role': r['role'], 'path': r['path'], **compact_metrics(r['metrics'])}
        rows.append(row)
    ranked_df = pd.DataFrame(rows)
    ranked_df.to_csv(OUTPUT_ROOT / 'test_metric_ranking.csv', index=False)

    print('Best overall:', selection_summary['best_overall'])
    print('Best classic:', selection_summary['best_classic'])
    print('Ablations:', selection_summary['ablations'])
    print('\nSaved ranking:', (OUTPUT_ROOT / 'test_metric_ranking.csv').resolve())
else:
    ranked = []
    ranked_df = pd.DataFrame()
    selection_summary = {
        'selection_metric': metric_name,
        'best_overall': None,
        'best_classic': None,
        'ablations': [],
        'skipped_reason': 'RUN_TEST_EVALUATION=False',
    }
    save_json(selection_summary, OUTPUT_ROOT / 'selection_summary.json')
    print('No metric ranking because RUN_TEST_EVALUATION=False.')

ranked_df

In [ ]:
# Save global metric visualizations.
fig_dir = OUTPUT_ROOT / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

if results:
    metric_cols = ['sample_f1', 'micro_f1', 'macro_f1', 'weighted_f1', 'micro_precision', 'micro_recall', 'hamming_loss']
    metric_long = ranked_df.melt(id_vars=['key', 'model_type', 'role'], value_vars=metric_cols, var_name='metric', value_name='value')

    plt.figure(figsize=(12, 5))
    sns.barplot(data=metric_long, x='metric', y='value', hue='key')
    plt.xticks(rotation=25, ha='right')
    plt.title('Test metrics for selected saved models')
    plt.tight_layout()
    metric_plot = fig_dir / 'test_metric_comparison.png'
    plt.savefig(metric_plot, dpi=170)
    plt.show()

    per_label_rows = []
    for r in results:
        for genre, vals in r['metrics'].get('per_label', {}).items():
            per_label_rows.append({'key': r['key'], 'genre': genre, **vals})
    per_label_df = pd.DataFrame(per_label_rows)
    per_label_df.to_csv(OUTPUT_ROOT / 'per_label_metrics.csv', index=False)

    plt.figure(figsize=(12, 8))
    sns.barplot(data=per_label_df, y='genre', x='f1', hue='key')
    plt.xlim(0, 1)
    plt.title('Per-label F1 on test set')
    plt.tight_layout()
    per_label_plot = fig_dir / 'per_label_f1_comparison.png'
    plt.savefig(per_label_plot, dpi=170)
    plt.show()

    print('Saved figures:')
    print(metric_plot.resolve())
    print(per_label_plot.resolve())
    print('Saved per-label table:', (OUTPUT_ROOT / 'per_label_metrics.csv').resolve())
else:
    per_label_df = pd.DataFrame()
    print('Skipped metric visualizations because RUN_TEST_EVALUATION=False.')

In [ ]:
# Run local XAI analysis for each selected model.
# For neural ablations, modality-specific methods are enabled/disabled to avoid meaningless or unsupported explainers.

def neural_xai_flags(entry):
    _, cfg, _, checkpoint, _ = load_neural_model(entry['path'], paths.metadata)
    modality = str(getattr(cfg, 'modality', checkpoint.get('config', {}).get('modality', 'multimodal'))).lower()
    image_encoder = str(getattr(cfg, 'image_encoder', checkpoint.get('config', {}).get('image_encoder', ''))).lower()
    multimodal = modality == 'multimodal'
    uses_text = modality in {'multimodal', 'text_only'}
    uses_image = modality in {'multimodal', 'image_only'}
    return {
        'modality': modality,
        'image_encoder': image_encoder,
        'enable_layer_integrated_gradients_text': uses_text,
        'enable_integrated_gradients_image': uses_image,
        'enable_gradcam': uses_image and image_encoder.startswith('resnet'),
        'enable_modality_ablation': multimodal,
        'enable_modality_shapley': multimodal,
        'enable_token_occlusion': uses_text,
        'enable_image_occlusion': uses_image,
    }


def choose_xai_indices(test_indices, labels, limit, strategy='balanced_support_bins', seed=42):
    test_indices = np.asarray(test_indices, dtype=np.int64)
    limit = min(int(limit), len(test_indices))
    rng = np.random.default_rng(seed)
    if limit <= 0:
        return np.asarray([], dtype=np.int64)
    if strategy == 'first':
        return test_indices[:limit]
    if strategy == 'random':
        return np.sort(rng.choice(test_indices, size=limit, replace=False))

    y_test = labels[test_indices]
    label_support = y_test.sum(axis=0).astype(int)
    available_labels = np.flatnonzero(label_support > 0)
    if len(available_labels) == 0:
        return np.sort(rng.choice(test_indices, size=limit, replace=False))

    if strategy == 'balanced_per_label':
        label_order = list(rng.permutation(available_labels))
    elif strategy == 'balanced_support_bins':
        # Spread requested examples across rare, medium, and common labels instead of taking test_idx[:N].
        sorted_labels = available_labels[np.argsort(label_support[available_labels])]
        positions = np.linspace(0, len(sorted_labels) - 1, num=min(limit, len(sorted_labels)), dtype=int)
        label_order = [int(sorted_labels[pos]) for pos in positions]
    else:
        raise ValueError(f'Unsupported XAI_SAMPLING_STRATEGY: {strategy}')

    selected = []
    used = set()
    for label_i in label_order:
        candidates = test_indices[y_test[:, label_i] == 1]
        candidates = [int(idx) for idx in candidates if int(idx) not in used]
        if not candidates:
            continue
        chosen = int(rng.choice(candidates))
        selected.append(chosen)
        used.add(chosen)
        if len(selected) >= limit:
            break

    if len(selected) < limit:
        remaining = [int(idx) for idx in test_indices if int(idx) not in used]
        fill = rng.choice(remaining, size=limit - len(selected), replace=False)
        selected.extend(int(idx) for idx in fill)

    return np.asarray(selected, dtype=np.int64)


def print_xai_sample_summary(indices, labels):
    print(f'Selected {len(indices)} base XAI samples using {XAI_SAMPLING_STRATEGY}:')
    rows = []
    for idx in indices:
        genres = [GENRE_LABELS[i] for i in np.flatnonzero(labels[int(idx)])]
        rows.append({'dataset_index': int(idx), 'true_genres': '|'.join(genres)})
    display(pd.DataFrame(rows))

xai_results = {}
xai_indices = choose_xai_indices(test_idx, y_all, XAI_LIMIT, XAI_SAMPLING_STRATEGY, XAI_RANDOM_SEED)
print_xai_sample_summary(xai_indices, y_all)

xai_root = OUTPUT_ROOT / 'xai'
xai_root.mkdir(parents=True, exist_ok=True)

if XAI_MODEL_KEYS is None:
    xai_model_manifest = model_manifest
else:
    wanted = set(XAI_MODEL_KEYS)
    xai_model_manifest = [entry for entry in model_manifest if entry['key'] in wanted]
    missing = wanted - {entry['key'] for entry in xai_model_manifest}
    if missing:
        raise ValueError(f'XAI_MODEL_KEYS contains unknown model keys: {sorted(missing)}')

if RUN_XAI:
    for entry in xai_model_manifest:
        print(f"\nRunning XAI for {entry['key']} on {len(xai_indices)} test samples")
        out_dir = xai_root / entry['key']
        if entry['model_type'] == 'classic':
            summary = explain_classic_samples(
                entry['path'],
                paths.hdf5,
                paths.metadata,
                xai_indices,
                out_dir,
                target_policy=XAI_TARGET_POLICY,
                target_top_k=XAI_TARGET_TOP_K,
                max_targets_per_sample=XAI_MAX_TARGETS_PER_SAMPLE,
                top_k=XAI_TOP_FEATURES,
                enable_modality_shapley=True,
                enable_experimental_set_explanation=True,
            )
        else:
            flags = neural_xai_flags(entry)
            print('Neural XAI flags:', flags)
            summary = explain_neural_samples(
                entry['path'],
                paths.hdf5,
                paths.metadata,
                xai_indices,
                out_dir,
                target_policy=XAI_TARGET_POLICY,
                target_top_k=XAI_TARGET_TOP_K,
                max_targets_per_sample=XAI_MAX_TARGETS_PER_SAMPLE,
                top_k=XAI_TOP_FEATURES,
                n_steps=XAI_N_STEPS,
                enable_layer_integrated_gradients_text=flags['enable_layer_integrated_gradients_text'],
                enable_integrated_gradients_image=flags['enable_integrated_gradients_image'],
                enable_gradcam=flags['enable_gradcam'],
                enable_modality_ablation=flags['enable_modality_ablation'],
                enable_modality_shapley=flags['enable_modality_shapley'],
                enable_token_occlusion=flags['enable_token_occlusion'],
                enable_image_occlusion=flags['enable_image_occlusion'],
                image_occlusion_grid=XAI_IMAGE_OCCLUSION_GRID,
                occlusion_batch_size=XAI_OCCLUSION_BATCH_SIZE,
                enable_experimental_set_explanation=flags['enable_modality_shapley'],
            )
        xai_results[entry['key']] = summary
        print('Saved XAI folder:', out_dir.resolve())
else:
    print('RUN_XAI=False, skipped local XAI.')

xai_results_path = OUTPUT_ROOT / 'xai_results_manifest.json'
xai_manifest = {}
for key, summary in xai_results.items():
    entry = next(item for item in xai_model_manifest if item['key'] == key)
    summary_name = 'classic_xai_summary.json' if entry['model_type'] == 'classic' else 'neural_xai_summary.json'
    xai_manifest[key] = {
        'model_type': entry['model_type'],
        'summary_path': str(xai_root / key / summary_name),
        'base_sample_indices': [int(idx) for idx in xai_indices],
        'sample_count': len(summary.get('samples', [])),
        'output_stats': summary.get('output_stats', {}),
    }
save_json(xai_manifest, xai_results_path)
print('Saved XAI manifest:', xai_results_path.resolve())

In [ ]:
# Aggregate XAI outputs into global summaries and plots.
if xai_results:
    global_xai_dir = OUTPUT_ROOT / 'xai_global'
    xai_summary = summarize_xai_outputs(xai_results, global_xai_dir)
    save_json(xai_summary, OUTPUT_ROOT / 'xai_global_summary.json')
    print('Saved global XAI summary:', (OUTPUT_ROOT / 'xai_global_summary.json').resolve())
    print('Saved global XAI folder:', global_xai_dir.resolve())
    pprint(xai_summary)
else:
    print('No XAI outputs to aggregate.')

In [ ]:
# Create a lightweight HTML report with saved output locations.
html_path = OUTPUT_ROOT / 'colab_test_xai_report.html'
figure_tags = []
for fig in sorted((OUTPUT_ROOT / 'figures').glob('*.png')):
    figure_tags.append(f"<figure><img src='{fig.relative_to(OUTPUT_ROOT).as_posix()}'><figcaption>{fig.name}</figcaption></figure>")
for fig in sorted((OUTPUT_ROOT / 'xai_global').glob('*.png')) if (OUTPUT_ROOT / 'xai_global').exists() else []:
    figure_tags.append(f"<figure><img src='{fig.relative_to(OUTPUT_ROOT).as_posix()}'><figcaption>{fig.name}</figcaption></figure>")

ranking_html = ranked_df.to_html(index=False, float_format=lambda v: f'{v:.4f}') if not ranked_df.empty else '<p>Metric evaluation skipped.</p>'
html = f"""
<!doctype html>
<html><head><meta charset='utf-8'><title>Colab Test XAI Report</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 24px; color: #18212b; }}
table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
th, td {{ border: 1px solid #ddd; padding: 6px; text-align: left; }}
figure {{ border: 1px solid #ddd; padding: 8px; margin: 12px 0; }}
img {{ max-width: 100%; }}
code {{ background: #f4f4f4; padding: 2px 4px; }}
</style></head><body>
<h1>Colab Test-Set Evaluation And XAI Report</h1>
<p>Output root: <code>{OUTPUT_ROOT.resolve()}</code></p>
<h2>Run Controls</h2>
<pre>{json.dumps({'RUN_TEST_EVALUATION': RUN_TEST_EVALUATION, 'RUN_XAI': RUN_XAI, 'XAI_LIMIT': XAI_LIMIT, 'XAI_SAMPLING_STRATEGY': XAI_SAMPLING_STRATEGY, 'XAI_RANDOM_SEED': XAI_RANDOM_SEED, 'XAI_MODEL_KEYS': XAI_MODEL_KEYS}, indent=2)}</pre>
<h2>Selection Summary</h2>
<pre>{json.dumps(selection_summary, indent=2)}</pre>
<h2>Metric Ranking</h2>
{ranking_html}
<h2>Figures</h2>
{''.join(figure_tags)}
<h2>Saved Files</h2>
<ul>
<li>Model manifest: <code>{(OUTPUT_ROOT / 'model_manifest.json').resolve()}</code></li>
<li>Combined metrics: <code>{(OUTPUT_ROOT / 'all_model_test_metrics.json').resolve()}</code></li>
<li>Ranking CSV: <code>{(OUTPUT_ROOT / 'test_metric_ranking.csv').resolve()}</code></li>
<li>Per-label CSV: <code>{(OUTPUT_ROOT / 'per_label_metrics.csv').resolve()}</code></li>
<li>Predictions folder: <code>{(OUTPUT_ROOT / 'predictions').resolve()}</code></li>
<li>XAI folder: <code>{(OUTPUT_ROOT / 'xai').resolve()}</code></li>
<li>Global XAI folder: <code>{(OUTPUT_ROOT / 'xai_global').resolve()}</code></li>
</ul>
</body></html>
"""
html_path.write_text(html, encoding='utf-8')
print('Saved HTML report:', html_path.resolve())
print('\nMain saved output locations:')
print('Output root:', OUTPUT_ROOT.resolve())
print('Metrics:', (OUTPUT_ROOT / 'all_model_test_metrics.json').resolve())
print('Predictions:', (OUTPUT_ROOT / 'predictions').resolve())
print('XAI:', (OUTPUT_ROOT / 'xai').resolve())
print('Report:', html_path.resolve())